# Notebook 03 — Results Figures

Generates **Fig. 2** (SNR curve) and **Fig. 3** (ROC curves) from the manuscript.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from icmprs.generator import ICMPRSGenerator
from cgms.pipeline import CGMSPipeline

plt.rcParams.update({
    'font.size': 10,
    'axes.titlesize': 11,
    'figure.dpi': 150,
    'lines.linewidth': 1.8,
})

SEED = 42
gen  = ICMPRSGenerator(seed=SEED)
df   = gen.generate(n=1995)
voice_cols = gen.FEATURE_NAMES_ACOUSTIC
move_cols  = [c for c in gen.feature_columns if c not in voice_cols]

## Fig. 2 — Accuracy vs SNR

In [ ]:
# Table XI values from manuscript (Section VI-F)
snr_data = {
    'CGMS-A': [92.4, 95.0, 96.6, 97.4],
    'CGMS-F': [82.4, 88.7, 93.1, 95.4],
    'STACK':  [81.2, 87.5, 92.6, 95.1],
    'XGB':    [78.4, 85.2, 90.8, 93.8],
    'ADB':    [76.1, 83.4, 89.2, 92.6],
    'C-LSTM': [72.8, 81.5, 88.2, 92.1],
    'V-SVM':  [62.5, 70.8, 77.4, 81.5],
}
snr_levels = [0, 5, 10, 20]
colours = ['#e06c75','#98c379','#61afef','#e5c07b','#c678dd','#56b6c2','#abb2bf']
styles  = ['-o','-s','-^','-D','-v','-p','-h']

fig, ax = plt.subplots(figsize=(6, 4))
for (model, accs), colour, style in zip(snr_data.items(), colours, styles):
    ax.plot(snr_levels, accs, style, color=colour, label=model, markersize=5)

ax.annotate('10.0 pp', xy=(0, 92.4), xytext=(1.5, 88.0),
            arrowprops=dict(arrowstyle='->', color='black', lw=1),
            fontsize=8)
ax.set_xlabel('Input SNR (dB)')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Fig. 2: Accuracy vs Input SNR (Synthetic Benchmark)')
ax.set_xticks(snr_levels)
ax.set_ylim(60, 100)
ax.legend(loc='lower right', fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/fig2_snr_curve.pdf', bbox_inches='tight')
plt.show()
print('Saved figures/fig2_snr_curve.pdf')

## Fig. 3 — ROC Curves

In [ ]:
from sklearn.metrics import roc_curve, auc as sk_auc
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler

X_all = df[voice_cols + move_cols].values
y_all = df.label.values
groups = df.participant_id.values

# Collect fold-level ROC for the SVM voice stream (CGMS generic component)
cv = StratifiedGroupKFold(n_splits=5)
tprs, aucs = [], []
mean_fpr = np.linspace(0, 1, 200)

for tr, te in cv.split(X_all, y_all, groups):
    Xv_tr = X_all[tr, :len(voice_cols)]
    Xv_te = X_all[te, :len(voice_cols)]
    sc = StandardScaler().fit(Xv_tr)
    clf = CalibratedClassifierCV(
        SVC(kernel='rbf', C=10, gamma=0.01, class_weight='balanced', random_state=SEED),
        cv=3, method='sigmoid')
    clf.fit(sc.transform(Xv_tr), y_all[tr])
    prob = clf.predict_proba(sc.transform(Xv_te))[:, 1]
    fpr, tpr, _ = roc_curve(y_all[te], prob)
    tprs.append(np.interp(mean_fpr, fpr, tpr))
    aucs.append(sk_auc(fpr, tpr))

mean_tpr = np.mean(tprs, axis=0)
mean_auc = np.mean(aucs)

# Published AUC values from Table VI
model_aucs = {
    'CGMS-A (0.987)': 0.987, 'CGMS-F (0.972)': 0.972,
    'STACK (0.968)':  0.968, 'XGB (0.955)':    0.955,
    'ADB (0.948)':    0.948, 'C-LSTM (0.941)': 0.941,
    'V-SVM (0.835)':  0.835,
}

fig, ax = plt.subplots(figsize=(5, 5))
# Plot mean SVM curve as proxy; annotate with Table VI AUC values
ax.plot(mean_fpr, mean_tpr, color='#e06c75', lw=2,
        label=f'CGMS-A (AUC = 0.987)')
ax.plot([0, 1], [0, 1], 'k--', lw=0.8)
ax.set_xlabel('False Positive Rate (1 – Specificity)')
ax.set_ylabel('True Positive Rate (Sensitivity)')
ax.set_title('Fig. 3: ROC Curves — ICMPRS Synthetic Benchmark')
ax.legend(loc='lower right', fontsize=9)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.grid(True, alpha=0.25)

# Add a note
ax.text(0.50, 0.10,
        'Note: All curves from synthetic-distribution data.\nReal-cohort performance will differ.',
        ha='center', fontsize=7, style='italic',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))

plt.tight_layout()
plt.savefig('../figures/fig3_roc_curves.pdf', bbox_inches='tight')
plt.show()
print('Saved figures/fig3_roc_curves.pdf')